# TinyLlama LoRA Fine-tune on Colab

Mirrors the local pipeline in this repo, but runs on a free Colab GPU (T4 ~16 GB).  
Expected time for 3.7k Q&A pairs, TinyLlama-1.1B, 3 epochs: **~10–30 minutes** end-to-end (vs. days on CPU).

## How to use
1. `File → Upload notebook` (or open from Drive) and pick this `.ipynb`.
2. `Runtime → Change runtime type → GPU` (T4 is free; L4/A100 if you have Pro).
3. Run the cells top to bottom. The **Upload dataset** cell prompts you for your `*.jsonl` file.
4. Last cell downloads the trained adapter (and optionally the merged HF model + GGUF) to your machine.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

Pin to the same library versions used locally so behavior matches.

In [ ]:
%%capture
!pip install -q -U \
    "transformers>=4.46" \
    "peft>=0.13" \
    "datasets>=3.0" \
    "accelerate>=1.0" \
    "trl>=0.11" \
    "sentencepiece>=0.2" \
    "safetensors>=0.4" \
    "bitsandbytes>=0.43" \
    "huggingface-hub>=0.25"

In [ ]:
import torch, transformers, peft, datasets, accelerate
print('torch       :', torch.__version__, 'cuda:', torch.cuda.is_available(),
      'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('transformers:', transformers.__version__)
print('peft        :', peft.__version__)
print('datasets    :', datasets.__version__)
print('accelerate  :', accelerate.__version__)

## 3. Upload your dataset

Pick the JSONL file you've been training on locally (the **approved** dataset from the Streamlit app, e.g. `merged-YYYYMMDD-HHMMSS.jsonl`). Each line must look like:

```json
{"messages": [
  {"role": "user", "content": "..."},
  {"role": "assistant", "content": "..."}
]}
```

In [ ]:
from google.colab import files
uploaded = files.upload()
DATASET_PATH = next(iter(uploaded.keys()))

# Quick sanity check.
import json
rows = 0
with open(DATASET_PATH, 'r', encoding='utf-8') as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        assert 'messages' in rec and len(rec['messages']) >= 2
        rows += 1
print(f'Loaded {rows} Q&A pairs from {DATASET_PATH}')

## 4. Configuration

Defaults below mirror the **recommended** settings from the local UI, dialed up for GPU (larger batch, fewer grad-accum steps, bf16). Change `EPOCHS` if you want a longer/shorter run.

In [ ]:
from pathlib import Path
from datetime import datetime

BASE_MODEL   = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'   # ← swap for Qwen, Llama-3.2-1B, etc.
RUN_ID       = f"tinyllama-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
OUTPUT_DIR   = Path('/content/runs') / RUN_ID
ADAPTER_DIR  = OUTPUT_DIR / 'adapter'
MERGED_DIR   = OUTPUT_DIR / 'merged'

# LoRA
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
TARGET_MODULES = None   # None → PEFT auto-detect; or ['q_proj','k_proj','v_proj','o_proj']

# Training loop (GPU-tuned)
EPOCHS              = 3        # 3 is usually plenty on a 3k–7k Q&A pair set; bump to 5–8 if loss still falling
LEARNING_RATE       = 3e-4
PER_DEVICE_BATCH    = 4        # T4: 4–8 is safe for TinyLlama; L4/A100: 8–16
GRAD_ACCUM_STEPS    = 1
MAX_SEQ_LENGTH      = 384
WARMUP_RATIO        = 0.03
WEIGHT_DECAY        = 0.0
LR_SCHEDULER        = 'cosine'
OPTIMIZER           = 'adamw_torch'
SEED                = 42

# Mixed precision: bf16 on Ampere+ (T4 supports fp16 not bf16; we pick automatically)
import torch
BF16_OK = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
USE_BF16 = BF16_OK
USE_FP16 = not BF16_OK and torch.cuda.is_available()
print('precision:', 'bf16' if USE_BF16 else ('fp16' if USE_FP16 else 'fp32'))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir:', OUTPUT_DIR)

## 5. Load base model + tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if USE_BF16 else (torch.float16 if USE_FP16 else torch.float32)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    device_map='auto',
    low_cpu_mem_usage=True,
)
model.config.use_cache = False

## 6. Tokenize dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset('json', data_files=DATASET_PATH, split='train')

def _format(ex):
    text = tokenizer.apply_chat_template(
        ex['messages'], tokenize=False, add_generation_prompt=False,
    )
    enc = tokenizer(text, truncation=True, max_length=MAX_SEQ_LENGTH,
                    padding=False, return_tensors=None)
    enc['labels'] = enc['input_ids'].copy()
    return enc

ds = ds.map(_format, remove_columns=ds.column_names, desc='tokenizing')
print('Train examples:', len(ds))

## 7. Attach LoRA adapter

In [ ]:
from peft import LoraConfig, get_peft_model

lora = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias='none', task_type='CAUSAL_LM',
    target_modules=TARGET_MODULES,
)
model = get_peft_model(model, lora)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## 8. Train

In [ ]:
from transformers import (Trainer, TrainingArguments,
                          DataCollatorForSeq2Seq)

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'trainer'),
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type=LR_SCHEDULER,
    optim=OPTIMIZER,
    logging_steps=10,
    save_strategy='no',
    report_to=[],
    bf16=USE_BF16,
    fp16=USE_FP16,
    seed=SEED,
    remove_unused_columns=False,
)

# Seq2Seq collator pads input_ids + labels (with -100 for ignore) — avoids the
# 'expected sequence of length N at dim 1' error in transformers 5.x.
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True,
                                  label_pad_token_id=-100, return_tensors='pt')

trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collator)
out = trainer.train()
print('Final loss:', out.training_loss)

## 9. Save the LoRA adapter

In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print('Adapter at:', ADAPTER_DIR)

## 10. (Optional) Merge LoRA into base → full HF model

Skip if you only need the adapter. Merging produces a ~2 GB folder you can convert to GGUF and load in Ollama / llama.cpp.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Reload the base model in fp16 for merging (the wrapped peft model is harder to merge cleanly).
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, low_cpu_mem_usage=True
)
merged = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
merged = merged.merge_and_unload()

MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))
print('Merged HF model at:', MERGED_DIR)

## 11. (Optional) Convert merged model → GGUF for Ollama

Builds llama.cpp on Colab and produces a `.gguf` file. Skip if you'll convert locally.

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
%pip install -q -r /content/llama.cpp/requirements.txt

GGUF_PATH = str(OUTPUT_DIR / f'{RUN_ID}.q8_0.gguf')
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_PATH} --outtype q8_0
print('GGUF at:', GGUF_PATH)

## 12. Download artifacts to your machine

Pick one or both:
- **Adapter only** (~30 MB): smallest, requires the base model on the other side.
- **GGUF** (~1.2 GB for q8_0): single file, plug straight into Ollama.

In [ ]:
import shutil
from google.colab import files

# Adapter as a zip
adapter_zip = shutil.make_archive(str(OUTPUT_DIR / 'adapter'), 'zip', str(ADAPTER_DIR))
files.download(adapter_zip)

# If you produced GGUF:
# files.download(GGUF_PATH)

## 13. Use it in Ollama (local)

After downloading `*.gguf`:

```bash
# Modelfile content (save as `Modelfile`):
# FROM ./tinyllama-<timestamp>.q8_0.gguf
# PARAMETER temperature 0.3
# PARAMETER top_p 0.9
# SYSTEM "You are a domain-specific assistant fine-tuned on the user's documents."

ollama create my-tinyllama-ft -f Modelfile
ollama run my-tinyllama-ft
```